# VL6180X: Filter Development

In [ ]:
import kalman_filter
import numpy as np
import plotly.graph_objects as go

# Measurement data
raw_data_no_improvement_str = "39;37;35;35;37;33;39;31;36;41;35;32;39;34;37;35;31;35;37;34;41;35;32;35;34;40;35;34;35;34;37;37;37;32;35;36;37;40;42;38;37;38;38;35;35;35;36;36;37;37;31;35;38;35;37;31;35;36;32"
raw_data_increased_averaging_str = "35;35;34;33;33;35;32;35;35;34;34;35;34;33;36;34;35;33;34;35;33;34;36;34;35;34;34;34;33;35;32;34;35;32;34;35;34;35;36;33;31;36;34;34;33;34;35;36;34;33;34;34;34;36;35;32;34;36;33"

raw_data_increased_averaging = np.array([int(val) for val in raw_data_increased_averaging_str.split(";")])
raw_data_no_improvement = np.array([int(val) for val in raw_data_no_improvement_str.split(";")])

# Run the 1D Kalman Filter (R = 1.2, Q = 0.01)
kf = kalman_filter.KalmanFilter1D(measurement_noise=1.2, process_noise=0.01)
filtered_data = np.array([kf.update(val) for val in raw_data_increased_averaging])

# Print comparison metrics 
print(
    f"Raw Data -> Mean: {np.mean(raw_data_no_improvement):.2f} mm | SD:"
    f" {np.std(raw_data_no_improvement):.2f} mm"
)
print(
    f"Raw Data + Increased Averaging -> Mean: {np.mean(raw_data_increased_averaging):.2f} mm | SD:"
    f" {np.std(raw_data_increased_averaging):.2f} mm"
)
print(
    f"Raw Data + Increased Averaging + Kalman Filter -> Mean: {np.mean(filtered_data[10:]):.2f} mm | SD:"
    f" {np.std(filtered_data[10:]):.2f} mm"
)

# Plot interactive comparison with Plotly
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=raw_data_no_improvement,
        mode="lines+markers",
        name="Raw Sensor Data",
        line=dict(color="rgba(150, 150, 150, 0.6)", width=1.5, dash="dot"),
        marker=dict(size=6),
    )
)

fig.add_trace(
    go.Scatter(
        y=filtered_data,
        mode="lines+markers",
        name="1D Kalman Filter (R=1.2, Q=0.01)",
        line=dict(color="#1f77b4", width=3),
        marker=dict(size=5),
    )
)

fig.update_layout(
    title="VL6180X Distance Measurements: Raw vs. 1D Kalman Filter + Increased Hardware Averaging",
    xaxis_title="Sample Index",
    yaxis_title="Distance (mm)",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(
        xanchor="right",
        yanchor="top"
    )
)


mean_val = np.mean(raw_data_no_improvement)
std_val = np.std(raw_data_no_improvement)  # Use ddof=1 for sample standard deviation
median_val = np.median(raw_data_no_improvement)
min_val, max_val = np.min(raw_data_no_improvement), np.max(raw_data_no_improvement)


# Add shaded ±1 Standard Deviation band
fig.add_hrect(
    y0=mean_val - std_val,
    y1=mean_val + std_val,
    fillcolor="LightSalmon",
    opacity=0.25,
    line_width=0,
    annotation_text="±1 SD",
    annotation_position="top left",
)

# Add horizontal lines for Mean and Median
fig.add_hline(
    y=mean_val,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean: {mean_val:.2f}",
    annotation_position="top right",
)

fig.show()
